# Comment Score Analysis

This notebook analyzes score distributions and text evaluation fields from `comment_scores.csv` and `comment_scores.jsonl`.

It is designed to compare human vs AI comments once AI-generated comments are added and labeled via `source_label`.

In [ ]:
# Install dependencies (run once)
%pip install -q pandas matplotlib seaborn bertopic sentence-transformers umap-learn hdbscan scikit-learn

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

BASE_DIR = Path.cwd()
ANALYSIS_DIR = BASE_DIR
CSV_PATH = ANALYSIS_DIR / "comment_scores.csv"
JSONL_PATH = ANALYSIS_DIR / "comment_scores.jsonl"

## Load Data

Use the CSV for tabular analysis and the JSONL for raw text fields if needed.

In [ ]:
df = pd.read_csv(CSV_PATH)

def load_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    for line in path.read_text(encoding="utf-8").splitlines():
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return rows

raw_rows = load_jsonl(JSONL_PATH)

len(df), len(raw_rows)

## Quick Overview

Check label counts and missing values.

In [ ]:
df["source_label"].value_counts(dropna=False)

df.isna().sum().sort_values(ascending=False).head(10)

## Score Distributions

Compare overall and per-criterion scores by label.

In [ ]:
score_cols = [c for c in df.columns if c.startswith("score_")]

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if "score_overall" in df.columns:
    sns.histplot(data=df, x="score_overall", hue="source_label", multiple="stack", bins=5, ax=axes[0])
    axes[0].set_title("Overall Score Distribution")
else:
    axes[0].axis("off")

sns.boxplot(data=df, x="source_label", y="score_relevance", ax=axes[1])
axes[1].set_title("Relevance by Source")

plt.tight_layout()

## Summary Table by Source

Mean scores by label (human vs AI).

In [ ]:
df.groupby("source_label")[score_cols].mean().round(2)

In [ ]:
rationale_cols = [c for c in df.columns if c.startswith("rationale_")]
labels = df["source_label"].fillna("unknown").unique().tolist()

MIN_DOCS = 10


def prep_texts(series):
    return [s for s in series.dropna().astype(str).tolist() if s.strip()]


def fit_topics(texts):
    if len(texts) < MIN_DOCS:
        return None
    vectorizer = CountVectorizer(stop_words="english", min_df=2)
    model = BERTopic(vectorizer_model=vectorizer, verbose=False)
    model.fit(texts)
    return model


topic_models = {}

topic_summaries = {}
for col in rationale_cols:
    topic_summaries[col] = {}
    for label in labels:
        subset = df[df["source_label"].fillna("unknown") == label]
        texts = prep_texts(subset[col])
        model = fit_topics(texts)
        if model is None:
            topic_summaries[col][label] = pd.DataFrame({
                "note": [f"Not enough documents ({len(texts)})"]
            })
            continue
        topic_models[(col, label)] = model
        topic_summaries[col][label] = model.get_topic_info().head(10)

for col in rationale_cols:
    print(f"\n=== {col} ===")
    for label in labels:
        print(f"\n-- {label} --")
        display(topic_summaries[col][label])

## Topic Modeling by Rationale (BERTopic)

This fits a separate topic model for each rationale field and compares topics between human and AI labels.

## Best and Worst Examples

Show comments with the highest and lowest overall scores.

In [ ]:
if "score_overall" in df.columns:
    top = df.sort_values("score_overall", ascending=False).head(5)
    bottom = df.sort_values("score_overall", ascending=True).head(5)
    display(top[["file", "source_label", "policy_id", "score_overall", "overall_summary"]])
    display(bottom[["file", "source_label", "policy_id", "score_overall", "overall_summary"]])